# Assignment Module 2: Aircraft Classification

The goal of this assignment is to implement a neural network that classifies images of 100 aircraft model variants from the [Fine-Grained Visual Classification of Aircraft (**FGVC-Aircraft**) dataset](https://www.robots.ox.ac.uk/~vgg/data/fgvc-aircraft/). The assignment is divided into two parts: first, you will be asked to implement your own neural network for image classification from scratch; then, you will fine-tune a pretrained network provided by PyTorch.

![](https:///raw.githubusercontent.com/CVLAB-Unibo/ipcv-assignment-2/master/fgvc_aircraft_variants.svg)

## Dataset

Download and acces the dataset through its official [PyTorch `FGVCAircraft` class](https://docs.pytorch.org/vision/main/generated/torchvision.datasets.FGVCAircraft.html) (by setting its constructor argument `annotation_level` to `'variant'`).

## Part 1: design your own network

Your goal is to implement a convolutional neural network for image classification and train it from scratch on `FGVCAircraft`. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~50%. You are free to achieve this however you want, except for a few rules you must follow:

- Compile this notebook by displaying the results obtained by the best model you found throughout your experimentation; then show how, by removing some of its components, its performance drops. In other words, do an *ablation study* to prove that your design choices have a positive impact on the final result.

- Do not instantiate an off-the-self PyTorch network. Instead, construct your network as a composition of existing PyTorch layers. In more concrete terms, you can use e.g. `torch.nn.Linear`, but you cannot use e.g. `torchvision.models.alexnet`.

- Show your results and ablations with plots, tables, images, etc. — the clearer, the better.

Don't be too concerned with your model performance: the ~50% is just to give you an idea of when to stop. Keep in mind that a thoroughly justified model with lower accuracy will be rewarded more points than a poorly experimentally validated model with higher accuracy.

## Part 2: fine-tune an existing network

Your goal is to fine-tune a pretrained ResNet-18 model on `FGVCAircraft`. Use the implementation provided by PyTorch, i.e. the opposite of part 1. Specifically, use the PyTorch ResNet-18 model pretrained on ImageNet-1K (V1). Divide your fine-tuning into two parts:

2A. First, fine-tune the ResNet-18 with the same training hyperparameters you used for your best model in part 1.

2B. Then, tweak the training hyperparameters to increase the accuracy on the test split. Justify your choices by analyzing the training plots and/or citing sources that guided you in your decisions — papers, blog posts, YouTube videos, or whatever else you may find useful. You should consider yourselves satisfied once you obtain a classification accuracy on the test split of ~70%.

# Solution

## Table of contents

- [Import dependencies](#Import-dependencies)
- [Runtime settings](#Runtime-settings)
- [Show Report Images](#show-report-images)
- [Dataset and transforms](#Dataset-and-transforms)
- [Show dataset preview](#Show-dataset-preview)
- [Training functions](#Training-functions)
- [Run training](#Run-training)
- [Custom CNN model](#Custom-CNN-model)
  - [Best model](#Best-model)
  - [Part 1 ablation study](#Part-1-ablation-study)
    - [Remove augmentation](#Remove-augmentation)
    - [Make augmentation heavy](#Make-augmentation-heavy)
    - [Remove mixup](#Remove-mixup)
    - [Remove label smoothing](#Remove-label-smoothing)
    - [Remove dropout](#Remove-dropout)
    - [Increase dropout](#Increase-dropout)
    - [Remove Batch Normalization](#Remove-Batch-Normalization)
    - [Decrease learning rate](#Decrease-learning-rate)
  - [Complete list of experiments and their results](#Complete-list-of-experiments-and-their-results)
- [Part 2](#Part-2)
  - [Load pretrained ResNet-18 model](#Load-pretrained-ResNet-18-model)
  - [Part 2A: ResNet-18 with Part 1 hyperparameters](#Part-2A:-ResNet-18-with-Part-1-hyperparameters)
    - [Freeze backbone](#Freeze-backbone)
    - [Train the whole model](#Train-the-whole-model)
  - [Part 2B: full fine-tuning](#Part-2B:-full-fine-tuning)
  - [Part 2B ablation study](#Part-2B-ablation-study)
    - [Freeze the backbone](#Part-2B-freeze-the-backbone)
    - [Remove augmentation](#Part-2B-remove-augmentation)
    - [Make augmentation heavy](#Part-2B-make-augmentation-heavy)
    - [Remove mixup](#Part-2B-remove-mixup)
    - [Remove label smoothing](#Part-2B-remove-label-smoothing)
    - [Increase dropout](#Part-2B-increase-dropout)
    - [Use differential backbone and head learning rates](#Part-2B-differential-learning-rates)
  - [Final comparison](#Final-comparison)
  - [Result of ResNet training](#Result-of-ResNet-training)
- [Final remarks](#Final-remarks)

You can find the solution to this assignment in the following [GitHub repository](https://github.com/mohamadch91/UNIBO_IPCV_Image_classification).
Repository contains the code for both parts of the assignment, also includes all training history, plots, and results. The code is organized in a way that allows you to easily reproduce the results and run your own experiments.


## Import dependencies


In [ ]:
# Standard-library utilities for copying configurations, serializing results, seeding, timing, and paths.
import copy
import json
import random
import time
from pathlib import Path
from typing import Any, Callable, Dict, List, Optional
from PIL import Image

# Optional Google Colab integration. It remains disabled for local execution.
# from google.colab import drive
import gdown

# Plotting, numerical processing, tabular results, and PyTorch training primitives.
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

# Confusion-matrix calculation and visualization.
from sklearn.metrics import confusion_matrix,ConfusionMatrixDisplay

# Notebook-aware progress bars.
from tqdm.notebook import tqdm

# Optimizer, scheduler, data, transform, and pretrained-model utilities.
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR, CosineAnnealingLR
from torch.utils.data import DataLoader, Subset,ConcatDataset
from torchvision import datasets, transforms as T
from torchvision.models import resnet18, ResNet18_Weights
from torchsummary import summary

## Runtime settings


In [ ]:
def fix_random(seed: int) -> None:
    """Make NumPy, Python, and PyTorch randomness reproducible.

    Args:
        seed: Integer seed shared by all random-number generators.

    Returns:
        None. The function updates global random-number-generator and cuDNN settings.
    """
    # Seed every random-number generator used by the notebook.
    np.random.seed(seed)
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    # Prefer deterministic cuDNN kernels over benchmark-selected kernels.
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True



# Apply the same seed before data loading, initialization, and training.
fix_random(seed=42)

# Select CUDA first, then Apple MPS, and otherwise fall back to CPU.
# I used MPS for training.
device = "cpu"
if torch.cuda.is_available():
    print("All good, a GPU is available")
    device = torch.device("cuda:0")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("Using Apple Silicon GPU through MPS")
    device = torch.device("mps")
else:
    print("Using CPU")

# Create the local data and output paths. The commented alternatives support Google Drive.
# drive.mount('/content/drive',force_remount=True)
# path_default = Path("/content/drive/MyDrive/IPCV_E2/")
path_default = Path(".")

path_data = path_default / Path("data")
path_outputs = path_default / Path("outputs")
path_ckpts = path_default / path_outputs / "ckpts"
path_histories = path_outputs / "histories"
path_plots = path_outputs / "plots"
path_summaries = path_outputs / "summaries"

# Create every required directory if it does not already exist.
if not path_default.exists():
    path_default.mkdir()
if not path_data.exists():
    path_data.mkdir()
if not path_outputs.exists():
    path_outputs.mkdir()
if not path_ckpts.exists():
    path_ckpts.mkdir()
if not path_histories.exists():
    path_histories.mkdir()
if not path_plots.exists():
    path_plots.mkdir()
if not path_summaries.exists():
    path_summaries.mkdir()

# Global flags control downloading, training, previewing, and optional confusion matrices.
DOWNLOAD_DATA = True
RUN_TRAINING = True
SHOW_DATASET_PREVIEW = True
SHOW_CONFUSION_MATRIX = False
PIN_MEMORY = torch.cuda.is_available()

# gdown ids for showing images

drive_links = {
    "resnet_png" : "https://drive.google.com/file/d/1r-C6iwn_pY-pWrcgMse1nj8Gku9Z6KEM",
    "build_png" : "https://drive.google.com/file/d/1YJlcmXNiENyZU5HES9ygZkiQhMPERQS",
    "base_png" : "https://drive.google.com/file/d/1yn-NLvvFtrfO9TfVYFSkmK_8D2S3PRK2",
    "base_build_png" : "https://drive.google.com/file/d/1SSPJ0I46ApzN1-ucl7Ar7IhMYAKqnt6-",
    "best_model": "https://drive.google.com/file/d/1gl1JmC5XXHioqj-Rbprn_qr8eatiW4CE",
    "resnet_best" : "",
}

# The shared configuration contains image processing, regularization, architecture,
# optimizer, early-stopping, and training hyperparameters.
cfg = {
    # Image, crop, and resize sizes.
    "resize_size": 256,
    "crop_size": 224,
    "image_size": 224,

    # Input augmentation and regularization settings.
    "augmentation": "light",
    "dropout": 0.15,
    "mixup_alpha": 0.85,
    "mixup_prob": 0.75,
    "label_smoothing": 0.1,

    # Custom-model architecture settings.
    "channels": [32, 64, 128, 256],
    "use_batch_norm": True,

    # ResNet-specific fine-tuning settings.
    "freeze_backbone": False,
    "head_lr": None,

    # AdamW and learning-rate settings.
    "lr": 3e-3,
    "wd": 5e-4,

    # Validation-accuracy early-stopping settings.
    "early_stop_patience": 12,
    "min_delta": 1e-4,

    # Mini-batch and epoch settings.
    "batch_size": 64,
    "num_epochs": 70,
}

# Display the complete configuration as a readable one-column table.
pd.DataFrame.from_dict(cfg, orient="index", columns=["value"])

# Show Report Images

In [ ]:
def download_and_show_image(
    name: str,

):
    """Download and display an image selected by name.

    Args:
        name: Key identifying the image, such as "resnet" or "base".
        

    Returns:
        A tuple containing:
            - image: The downloaded PIL Image.
    

    Raises:
        KeyError: If name does not exist in drive_links.
        RuntimeError: If Google Drive could not download the file.
        ValueError: If the downloaded file is not a readable image.
    """
    # Ensure the requested image name exists.
    if name not in drive_links:
        available_names = ", ".join(sorted(drive_links))
        raise KeyError(
            f"Unknown image name: {name!r}. "
            f"Available names: {available_names}"
        )


   

    downloaded_path = gdown.download(
            url=drive_links[name],
            output=str(name),
            quiet=False,
            fuzzy=True,
        )
    image = Image.open(name)

    # Open and verify that the downloaded file is a valid image.
    try:
        image.load()
    except Exception as error:
        raise ValueError(
            f"The downloaded {name!r} file is not a readable image."
        ) from error

    # Display the image without axes.
    plt.figure(figsize=(14, 8))
    plt.imshow(image)
    plt.title(name.replace("_", " ").title())
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    return image

## Dataset and transforms

In [ ]:
# ImageNet normalization is used by both the custom CNN and pretrained ResNet-18.
mean_image_net = [0.485, 0.456, 0.406]
std_image_net = [0.229, 0.224, 0.225]


def get_train_transform(cfg: Dict[str, Any]) -> T.Compose:
    """Build the training transform selected by the experiment configuration.

    Args:
        cfg: Experiment dictionary containing ``augmentation``, image sizes.
           

    Returns:
        A composed torchvision transform that converts an input PIL image to a
        normalized training tensor.

    Raises:
        ValueError: If ``cfg['augmentation']`` is not a supported policy.
    """
    # Construct exactly one augmentation policy before adding optional erasing.
    if cfg["augmentation"] == "none":
        transform_list = [
            T.Resize((cfg["crop_size"], cfg["crop_size"])),
            T.ToTensor(),
            T.Normalize(mean_image_net, std_image_net)
        ]
    elif cfg["augmentation"] == "light":
        transform_list = [
            T.Resize(cfg["resize_size"]),
            T.RandomCrop(cfg["crop_size"]),
            T.RandomHorizontalFlip(),
            T.ToTensor(),
            T.Normalize(mean_image_net, std_image_net)
        ]
    elif cfg["augmentation"] == "strong":
        transform_list = [
            T.RandomResizedCrop(
                cfg["crop_size"],
                scale=(0.88, 1.0),
                ratio=(0.95, 1.05),
            ),
            T.RandomHorizontalFlip(p=0.5),
            T.RandomRotation(8),
            T.ColorJitter(
                brightness=0.08,
                contrast=0.08,
                saturation=0.08,
                hue=0.01,
            ),
            T.ToTensor(),
            T.Normalize(mean_image_net, std_image_net)
        ]
    elif cfg["augmentation"] == "resnet_strong":
        transform_list = [
            T.RandomResizedCrop(cfg["crop_size"], scale=(0.60, 1.00), ratio=(0.80, 1.20)),
            T.RandomHorizontalFlip(),
            T.RandAugment(num_ops=2, magnitude=9),
            T.ToTensor(),
            T.Normalize(mean_image_net, std_image_net)
        ]
    else:
        raise ValueError(f"Unknown augmentation policy: {cfg['augmentation']}")

    

    return T.Compose(transform_list)


def get_eval_transform(cfg: Dict[str, Any]) -> T.Compose:
    """Build the deterministic validation/test preprocessing pipeline.

    Args:
        cfg: Experiment dictionary containing ``resize_size`` and ``crop_size``.

    Returns:
        A composed transform that resizes, center-crops, converts, and normalizes
        one evaluation image.
    """
    return T.Compose([
        T.Resize(cfg["resize_size"]),
        T.CenterCrop(cfg["crop_size"]),
        T.ToTensor(),
        T.Normalize(mean_image_net, std_image_net)
    ])


def mixup(x, y, num_classes, alpha=0.2, prob=0.5):
    """Optionally mix a batch of images and its one-hot class targets.

    Args:
        x: Input image tensor with batch dimension first.
        y: Integer class-label tensor for the batch.
        num_classes: Total number of output classes.
        alpha: Symmetric beta-distribution parameter used to sample the mixing weight.
        prob: Probability of applying mixup to this batch.

    Returns:
        A pair ``(images, targets)``. Images are either original or mixed; targets
        are always floating-point one-hot or mixed class distributions.
    """
    # Convert labels even when the stochastic branch skips mixing.
    y_onehot = F.one_hot(y, num_classes).float()
    if np.random.rand() > prob or alpha <= 0:
        return x, y_onehot  # Skip mixing this batch.

    # Mix every sample with a randomly permuted sample from the same batch.
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    x = lam * x + (1 - lam) * x[idx]
    y_mixed = lam * y_onehot + (1 - lam) * y_onehot[idx]
    return x, y_mixed


def get_datasets(cfg: Dict[str, Any]):
    """Load the official train, validation, and test FGVC-Aircraft splits.

    Args:
        cfg: Experiment configuration used to construct train/evaluation transforms.

    Returns:
        ``(data_train, data_val, data_test, classes)`` where the first three items
        are torchvision datasets and ``classes`` is the ordered variant-name list.
    """
    # Training uses stochastic augmentation; validation and test are deterministic.
    data_train = datasets.FGVCAircraft(
        root=path_data,
        split="train",
        annotation_level="variant",
        transform=get_train_transform(cfg),
        download=DOWNLOAD_DATA,
    )
    data_val = datasets.FGVCAircraft(
        root=path_data,
        split="val",
        annotation_level="variant",
        transform=get_eval_transform(cfg),
        download=DOWNLOAD_DATA
    )
    data_test = datasets.FGVCAircraft(
        root=path_data,
        split="test",
        annotation_level="variant",
        transform=get_eval_transform(cfg),
        download=DOWNLOAD_DATA
    )

    # The class order is shared by all three official splits.
    classes = data_train.classes
    return data_train, data_val, data_test, classes


def get_loaders(data_train, data_val, data_test, cfg: Dict[str, Any]):
    """Wrap the three datasets in mini-batch data loaders.

    Args:
        data_train: Training dataset; shuffled once per epoch.
        data_val: Validation dataset; kept in deterministic order.
        data_test: Test dataset; kept in deterministic order.
        cfg: Experiment dictionary containing ``batch_size``.

    Returns:
        ``(loader_train, loader_val, loader_test)`` as PyTorch DataLoader objects.
    """
    loader_train = DataLoader(
        data_train,
        batch_size=cfg["batch_size"],
        shuffle=True,
        pin_memory=PIN_MEMORY
    )
    loader_val = DataLoader(
        data_val,
        batch_size=cfg["batch_size"],
        shuffle=False,
        pin_memory=PIN_MEMORY
    )
    loader_test = DataLoader(
        data_test,
        batch_size=cfg["batch_size"],
        shuffle=False,
        pin_memory=PIN_MEMORY
    )
    return loader_train, loader_val, loader_test

## Show dataset preview

In [ ]:
class NormalizeInverse(T.Normalize):
    """Invert channel-wise normalization so tensors can be displayed as images."""

    def __init__(self, mean: List[float], std: List[float]) -> None:
        """Create an inverse normalization transform.

        Args:
            mean: Channel means used by the forward normalization.
            std: Channel standard deviations used by the forward normalization.

        Returns:
            None. The initialized object is a callable torchvision transform.
        """
        # Algebraically convert inverse normalization into Normalize parameters.
        mean = torch.as_tensor(mean)
        std = torch.as_tensor(std)
        std_inv = 1 / (std + 1e-7)
        mean_inv = -mean * std_inv
        super().__init__(mean=mean_inv, std=std_inv)

    def __call__(self, tensor):
        """Return a denormalized clone without modifying the input tensor.

        Args:
            tensor: Normalized image tensor.

        Returns:
            A cloned tensor mapped back toward the display domain.
        """
        return super().__call__(tensor.clone())


def show_grid(dataset, classes: List[str], process: Callable = None) -> None:
    """Display ten random labeled samples from a dataset.

    Args:
        dataset: Indexable dataset returning ``(image, integer_label)`` pairs.
        classes: Ordered class names used to convert labels to titles.
        process: Optional image transform, such as inverse normalization, applied
            immediately before display.

    Returns:
        None. The function renders a Matplotlib figure in the notebook.
    """
    fig = plt.figure(figsize=(15, 6))
    indices_random = np.random.randint(10, size=10, high=len(dataset))

    # Add one image and its variant label to each position in a 2 x 5 grid.
    for count, idx in enumerate(indices_random):
        fig.add_subplot(2, 5, count + 1)
        image, label = dataset[idx]
        title = classes[label]
        plt.title(title, fontsize=9)
        image_processed = process(image) if process is not None else image
        plt.imshow(T.ToPILImage()(image_processed))
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# Build the inverse transform once and optionally preview the validation split.
denormalize = NormalizeInverse(mean_image_net, std_image_net)

if SHOW_DATASET_PREVIEW:
    data_train_preview, data_val_preview, data_test_preview, classes = get_datasets(cfg)
    print(f"# Train samples = {len(data_train_preview)}")
    print(f"# Val samples = {len(data_val_preview)}")
    print(f"# Test samples = {len(data_test_preview)}")
    print(f"# Classes = {len(classes)}")
    show_grid(data_val_preview, classes, process=denormalize)

## Training functions

this section contains the functions which are used to calculate accuracy,loss,
model parameteres, get optimizer, scheduler, and a class Trainer for training the model.


In [ ]:
def ncorrect(scores, y):
    """Count correctly classified samples in one batch.

    Args:
        scores: Model logits with shape ``(batch_size, num_classes)``.
        y: Integer ground-truth labels with shape ``(batch_size,)``.

    Returns:
        A scalar tensor containing the number of correct top-1 predictions.
    """
    y_hat = torch.argmax(scores, -1)
    return (y_hat == y).sum()


def accuracy(scores, y):
    """Calculate top-1 accuracy for one batch.

    Args:
        scores: Model logits with shape ``(batch_size, num_classes)``.
        y: Integer ground-truth labels.

    Returns:
        A scalar tensor equal to correct predictions divided by batch size.
    """
    correct = ncorrect(scores, y)
    return correct.true_divide(y.shape[0])


def count_parameters(model: nn.Module, trainable_only: bool = False) -> int:
    """Count model parameters.

    Args:
        model: PyTorch module whose parameters are counted.
        trainable_only: When True, exclude parameters with ``requires_grad=False``.

    Returns:
        Total number of scalar parameters satisfying the selected condition.
    """
    if trainable_only:
        return sum(p.numel() for p in model.parameters() if p.requires_grad)
    return sum(p.numel() for p in model.parameters())


def get_optimizer(model: nn.Module, cfg: Dict[str, Any]):
    """Construct AdamW, optionally with separate backbone and classifier rates.

    Args:
        model: Model whose trainable parameters will be optimized.
        cfg: Experiment dictionary containing ``lr``, ``wd``, and optional ``head_lr``.

    Returns:
        A configured ``torch.optim.AdamW`` optimizer.
    """
    params = []

    # When head_lr is set, separate classifier parameters from the backbone.
    if cfg.get("head_lr") is not None:
        backbone_params = []
        head_params = []
        for name, p in model.named_parameters():
            if not p.requires_grad:
                continue
            if name.startswith("fc") or name.startswith("classifier"):
                head_params.append(p)
            else:
                backbone_params.append(p)

        if len(backbone_params) > 0:
            params.append({"params": backbone_params, "lr": cfg["lr"]})
        if len(head_params) > 0:
            params.append({"params": head_params, "lr": cfg["head_lr"]})
    else:
        # A single parameter group uses cfg['lr'] for every trainable parameter.
        params = [p for p in model.parameters() if p.requires_grad]

    return AdamW(params, lr=cfg["lr"], weight_decay=cfg["wd"])


class Trainer:
    """Train, validate, checkpoint, and record one classification experiment."""

    def __init__(
            self,
            model: nn.Module,
            train_loader: DataLoader,
            val_loader: DataLoader,
            device: torch.device,
            cfg: Dict[str, Any]
        ) -> None:
        """Initialize model state, AdamW, OneCycleLR, history, and early stopping.

        Args:
            model: Classification network to optimize.
            train_loader: Mini-batches used for gradient updates.
            val_loader: Mini-batches used for model selection and early stopping.
            device: CPU, CUDA, or MPS device on which computation runs.
            cfg: Full experiment configuration.

        Returns:
            None. Training starts only when ``train`` is called.
        """
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        self.cfg = cfg
        self.num_epochs = cfg["num_epochs"]

        # Move the model once, then create the optimizer and per-step schedule.
        self.model = model.to(device)
        self.optimizer = get_optimizer(self.model, cfg)
        max_lr = [group["lr"] for group in self.optimizer.param_groups]
        print(f"Max learning rates for OneCycleLR: {max_lr}")
        self.scheduler = OneCycleLR(
            self.optimizer,
            max_lr=max_lr,
            total_steps=self.num_epochs * len(train_loader)
        )

        # Keep an in-memory copy of the best validation-accuracy parameters.
        self.step = 0
        self.best_acc = 0.0
        self.best_epoch = -1
        self.best_params = copy.deepcopy(self.model.state_dict())

        # Stop after the configured number of epochs without sufficient accuracy gain.
        self.best_val_acc = float(-1)
        self.patience = self.cfg.get("early_stop_patience", 20)
        self.min_delta = self.cfg.get("min_delta", 1e-4)
        self.epochs_without_improvement = 0

        # Store per-epoch metrics and the checkpoint destination for this run.
        self.history = []
        self.ckpt_path = path_ckpts / f"{cfg['run_name']}.pt"

    def eval(self, loader, return_cm=False):
        """Evaluate average cross-entropy, top-1 accuracy, and optionally confusion.

        Args:
            loader: Validation or test DataLoader.
            return_cm: When True, also collect a confusion matrix.

        Returns:
            ``(loss, accuracy)`` or ``(loss, accuracy, confusion_matrix)`` when
            ``return_cm`` is True. Loss and accuracy are dataset-level averages.
        """
        self.model.eval()

        total_loss = 0
        total_samples = 0
        total_acc = 0

        if return_cm:
            all_preds = []
            all_labels = []

        # Evaluation disables gradient tracking and accumulates sample-weighted totals.
        with torch.no_grad():
            for imgs, labels in loader:
                imgs = imgs.to(self.device)
                labels = labels.to(self.device)

                scores = self.model(imgs)
                loss = F.cross_entropy(scores, labels, reduction="sum")

                total_loss += loss.item()
                total_samples += imgs.size(0)
                total_acc += ncorrect(scores, labels).item()

                if return_cm:
                    preds = scores.argmax(dim=1)
                    all_preds.extend(preds.cpu().numpy())
                    all_labels.extend(labels.cpu().numpy())

        total_loss /= total_samples
        total_acc /= total_samples

        if return_cm:
            cm = confusion_matrix(all_labels, all_preds)
            return total_loss, total_acc, cm

        return total_loss, total_acc

    def train(self) -> None:
        """Run epoch training, validation, checkpointing, and early stopping.

        Args:
            None.

        Returns:
            None. Metrics are appended to ``history`` and the best parameters are
            restored into ``model`` before the method finishes.
        """
        for e in tqdm(range(self.num_epochs), desc="Epoch"):
            print(f"\nTraining epoch {e + 1}/{self.num_epochs}")

            self.model.train()
            train_loss = 0.0
            train_acc = 0
            train_samples = 0

            pbar = tqdm(self.train_loader, desc=f"Epoch {e + 1}/{self.num_epochs}")

            # Perform one optimizer and scheduler step for every training batch.
            for imgs, labels in pbar:
                imgs = imgs.to(self.device)
                labels = labels.to(self.device)

                imgs_mixed, labels_mixed = mixup(
                    imgs,
                    labels,
                    self.cfg.get("num_classes", 100),
                    alpha=self.cfg.get("mixup_alpha", 0.2),
                    prob=self.cfg.get("mixup_prob", 0.5),
                )

                scores = self.model(imgs_mixed)
                log_probs = F.log_softmax(scores, dim=1)

                # Smooth the one-hot or mixed targets before soft-target cross-entropy.
                if self.cfg.get("label_smoothing", 0.0) > 0:
                    num_classes = scores.shape[1]
                    smoothing = self.cfg["label_smoothing"]
                    labels_mixed = (
                        labels_mixed * (1 - smoothing)
                        + smoothing / num_classes
                    )

                loss = -(labels_mixed * log_probs).sum(dim=1).mean()

                # Backpropagate, clip gradients, update weights, then advance OneCycleLR.
                self.optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    self.model.parameters(), max_norm=5.0
                )
                self.optimizer.step()
                self.scheduler.step()

                # Report hard-label training accuracy while optimizing soft targets.
                train_loss += loss.item() * imgs.size(0)
                train_acc += ncorrect(scores, labels).item()
                train_samples += imgs.size(0)
                pbar.set_postfix(loss=f"{loss.item():.4f}")
                self.step += 1

            train_loss /= train_samples
            train_acc /= train_samples

            # Validate once per epoch and append a serializable metric record.
            val_loss, val_acc = self.eval(self.val_loader)
            current_lr = self.optimizer.param_groups[0]["lr"]
            self.history.append({
                "epoch": e + 1,
                "train_loss": train_loss,
                "train_acc": train_acc,
                "val_loss": val_loss,
                "val_acc": val_acc,
                "lr": current_lr,
            })

            print(
                f"Epoch {e + 1:03d}/{self.num_epochs} | "
                f"LR: {current_lr:.2e} | "
                f"Train Loss: {train_loss:.4f} | "
                f"Train Acc: {train_acc:.4f} | "
                f"Val Loss: {val_loss:.4f} | "
                f"Val Acc: {val_acc:.4f}"
            )

            # Save the model whenever validation accuracy reaches a new maximum.
            if val_acc > self.best_acc:
                self.best_acc = val_acc
                self.best_epoch = e + 1
                self.best_params = copy.deepcopy(self.model.state_dict())
                torch.save(self.best_params, self.ckpt_path)
                print(f"✓ Saved new best model (val_acc={val_acc:.4f})")

            # Count epochs without a validation-accuracy improvement of min_delta.
            if val_acc > self.best_val_acc + self.min_delta:
                self.best_val_acc = val_acc
                self.epochs_without_improvement = 0
            else:
                self.epochs_without_improvement += 1
                print(
                    f"Validation accuracy did not improve "
                    f"({self.epochs_without_improvement}/{self.patience})"
                )

                if self.epochs_without_improvement >= self.patience:
                    print("\nEarly stopping triggered.")
                    print(
                        f"Best validation accuracy: {self.best_acc:.4f} "
                        f"(epoch {self.best_epoch})"
                    )
                    break

        # Restore the selected checkpoint in memory before final test evaluation.
        if self.best_params is not None:
            self.model.load_state_dict(self.best_params)

        print(
            f"\nTraining complete. "
            f"Best validation accuracy: {self.best_acc:.4f} "
            f"at epoch {self.best_epoch}."
        )

## Run training
This section contains the code to run the training of any model and plot the training curves. and summary of the result to table, csv, json.

In [ ]:
def plot_confusion_matrix(cm, class_names=None, normalize=False, save_path=None):
    """Display and optionally save a confusion matrix.

    Args:
        cm: Square matrix of true-class by predicted-class counts.
        class_names: Optional ordered labels for matrix axes.
        normalize: When True, convert each true-class row to proportions.
        save_path: Optional filesystem destination for the PNG figure.

    Returns:
        None. The function renders a Matplotlib figure and may save it to disk.
    """
    # Row normalization makes classes comparable even if counts differ.
    if normalize:
        cm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
        cm = cm.clip(min=0)

    fig, ax = plt.subplots(figsize=(30, 30))
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names,
    )
    disp.plot(
        ax=ax,
        cmap="Blues",
        xticks_rotation=90,
        colorbar=True,
        include_values=False,
        values_format=".2f" if normalize else "d"
    )

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()


def run_experiment(cfg_exp: Dict[str, Any], get_model_fn: Callable):
    """Train and evaluate one reproducible experiment and persist its artifacts.

    Args:
        cfg_exp: Complete configuration including a unique ``run_name``.
        get_model_fn: Callable receiving ``(num_classes, cfg_exp)`` and returning
            an initialized PyTorch classification model.

    Returns:
        A result dictionary containing scalar metrics, copied configuration, model
        parameter counts, and the per-epoch history DataFrame.
    """
    # Re-seed, then construct the data pipeline and model for this experiment.
    fix_random(seed=42)
    data_train, data_val, data_test, classes = get_datasets(cfg_exp)
    loader_train, loader_val, loader_test = get_loaders(data_train, data_val, data_test, cfg_exp)

    model = get_model_fn(len(classes), cfg_exp)
    print(f"Run name = {cfg_exp['run_name']}")
    print(f"Total parameters = {count_parameters(model):,}")
    print(f"Trainable parameters = {count_parameters(model, trainable_only=True):,}")

    # Train the model and measure wall-clock training time in minutes.
    trainer = Trainer(
        model,
        loader_train,
        loader_val,
        device,
        cfg_exp
    )
    start_time = time.time()
    trainer.train()
    elapsed = (time.time() - start_time) / 60

    # Evaluate the selected checkpoint on test data, optionally collecting confusion.
    if SHOW_CONFUSION_MATRIX:
        test_loss, test_acc, cm = trainer.eval(loader_test, True)
        plot_confusion_matrix(
            cm,
            class_names=data_train.classes,
            normalize=True,
            save_path=path_plots / (cfg_exp["run_name"] + "_conf.png")
        )
    else:
        test_loss, test_acc = trainer.eval(loader_test)

    # Persist the epoch history independently so it can be analyzed without rerunning.
    history = pd.DataFrame(trainer.history)
    history.to_csv(path_histories / f"{cfg_exp['run_name']}.csv", index=False)

    result = {
        "run_name": cfg_exp["run_name"],
        "best_val_acc": trainer.best_acc,
        "best_epoch": trainer.best_epoch,
        "test_loss": test_loss,
        "test_acc": test_acc,
        "elapsed_min": elapsed,
        "total_parameters": count_parameters(trainer.model),
        "trainable_parameters": count_parameters(trainer.model, trainable_only=True),
        "cfg": copy.deepcopy(cfg_exp),
        "history": history
    }

    # JSON stores scalar/configuration data; the DataFrame remains in CSV and memory.
    result_to_save = copy.deepcopy(result)
    result_to_save.pop("history")
    with open(path_summaries / f"{cfg_exp['run_name']}_summary.json", "w") as f:
        json.dump(result_to_save, f, indent=2)

    print(f"Best val acc = {trainer.best_acc:.3f}")
    print(f"Test acc = {test_acc:.3f}")
    return result


def results_to_table(results: Dict[str, Dict[str, Any]]) -> pd.DataFrame:
    """Convert experiment dictionaries into a test-accuracy-sorted table.

    Args:
        results: Mapping from run names to ``run_experiment`` result dictionaries.

    Returns:
        A DataFrame with one row per experiment and selected metrics/configuration.
    """
    rows = []
    for run_name, result in results.items():
        rows.append({
            "run_name": run_name,
            "best_val_acc": result["best_val_acc"],
            "test_acc": result["test_acc"],
            "test_loss": result["test_loss"],
            "best_epoch": result["best_epoch"],
            "augmentation": result["cfg"]["augmentation"],
            "lr": result["cfg"]["lr"],
            "head_lr": result["cfg"].get("head_lr"),
            "wd": result["cfg"]["wd"],
            "dropout": result["cfg"].get("dropout"),
            "label_smoothing": result["cfg"].get("label_smoothing"),
            "freeze_backbone": result["cfg"].get("freeze_backbone"),
            "trainable_parameters": result["trainable_parameters"]
        })

    # Sorting puts the strongest recorded test result first.
    table = pd.DataFrame(rows)
    if len(table) > 0:
        table = table.sort_values("test_acc", ascending=False).reset_index(drop=True)
    return table


def plot_results(results: Dict[str, Dict[str, Any]], save_path: Path = None) -> None:
    """Plot training and validation accuracy for multiple experiments.

    Args:
        results: Mapping from run names to results containing history DataFrames.
        save_path: Optional filesystem destination for the combined PNG figure.

    Returns:
        None. The function renders the two-panel Matplotlib figure.
    """
    _, ax = plt.subplots(1, 2, figsize=(20, 12))

    # Each run contributes one training curve and one validation curve.
    for run_name, result in results.items():
        history = result["history"]
        ax[0].plot(history["epoch"], history["train_acc"], label=f"{run_name} train")
        ax[1].plot(history["epoch"], history["val_acc"], label=f"{run_name} val")

    ax[0].set_title("Training accuracy")
    ax[1].set_title("Validation accuracy")
    for axis in ax:
        axis.set_xlabel("Epoch")
        axis.set_ylabel("Accuracy")
        axis.grid(alpha=0.3)
        axis.legend(fontsize=8)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=300)
    plt.show()

def download_model_and_show_prediction(
    model_url: str,
    dataset,
    get_model_fn: Callable,
    model_cfg: Dict[str, Any],
    checkpoint_path: str = "downloaded_model.pt",
    top_k: int = 5,
    device: Optional[torch.device] = None,
):
    """
    Download a model checkpoint from Google Drive and visualize:

        1. The original dataset image.
        2. An augmented version of the same image.
        3. The model's top-k output probabilities.

    The function reuses the notebook's existing:

        - Loaded FGVC-Aircraft dataset.
        - get_train_transform function.
        - denormalize transform.
        - Dataset class names.
        - Custom CNN or ResNet-18 model-building function.

    The dataset is NOT instantiated or downloaded again.

    Args:
        model_url:
            Public Google Drive URL of the saved PyTorch checkpoint.

        dataset:
            An FGVC-Aircraft dataset that is already loaded in the notebook,
            such as data_train_preview, data_val_preview, or data_test_preview.


        get_model_fn:
            Function used to construct the model architecture. Use either
            get_custom_cnn or get_resnet18_model.

        model_cfg:
            The exact configuration dictionary used to train the downloaded
            checkpoint.

        checkpoint_path:
            Local filename used to store the downloaded checkpoint.

        top_k:
            Number of highest-probability predictions shown in the plot.

        device:
            Device used for inference. When None, the function automatically
            selects CUDA if available and otherwise uses the CPU.

    Returns:
        model:
            The model containing the downloaded parameters.

        probabilities:
            A CPU tensor containing the probability of every class.

        predicted_class:
            Name of the class with the highest probability.
    """

    sample_index = np.random.randint(
        low=0,
        high=len(dataset),
    )
    

    # Check that at least one prediction should be displayed.
    if top_k <= 0:
        raise ValueError("top_k must be greater than zero.")

    # Automatically use the GPU when one is available.
    if device is None:
        device = torch.device(
            "cuda" if torch.cuda.is_available() else "cpu"
        )

    # Download the model checkpoint from Google Drive.
    #
    # fuzzy=True allows gdown to understand normal sharing URLs such as:
    # https://drive.google.com/file/d/FILE_ID/view?usp=sharing
    downloaded_path = gdown.download(
        url=model_url,
        output=checkpoint_path,
        quiet=False,
        fuzzy=True,
    )

    # gdown returns None if the download is unsuccessful.
    if downloaded_path is None:
        raise RuntimeError(
            "The model could not be downloaded. Make sure the Google Drive "
            "file is shared with 'Anyone with the link'."
        )

    # Reuse the class names already stored inside the loaded dataset.
    classes = dataset.classes

    # Build the same model architecture that was used during training.
    model = get_model_fn(
        len(classes),
        model_cfg,
    )

    # Load the downloaded checkpoint directly onto the selected device.
    #
    # weights_only=True is suitable for the notebook because Trainer saves
    # checkpoints using torch.save(model.state_dict(), checkpoint_path).
    checkpoint = torch.load(
        downloaded_path,
        map_location=device,
        weights_only=True,
    )

    # Support the raw state dictionary produced by the notebook as well as
    # common wrapped checkpoint formats.
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif isinstance(checkpoint, dict) and "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    # Load the trained parameters into the model architecture.
    #
    # strict=True reports an error if model_cfg or get_model_fn does not match
    # the architecture of the downloaded checkpoint.
    model.load_state_dict(
        state_dict,
        strict=True,
    )

    # Move the model to the selected device.
    model = model.to(device)

    # Enable evaluation mode so dropout is disabled and BatchNorm uses its
    # learned running statistics.
    model.eval()

    # Get the true label by using the already loaded dataset.
    #
    # The returned transformed tensor is not needed here because we want to
    # display the original image before any transformation.
    _, true_label = dataset[sample_index]

    # FGVCAircraft stores the paths of its original images in _image_files.
    #
    # Accessing this path allows us to recover the real PIL image without
    # constructing or downloading another dataset object.
    original_image_path = dataset._image_files[sample_index]

    # Open the original dataset image without applying the dataset transform.
    original_image = Image.open(original_image_path).convert("RGB")

    # Apply one stochastic training augmentation using the notebook's existing
    # augmentation function.
    #
    # The returned tensor is normalized using the ImageNet mean and standard
    # deviation defined in the notebook.
    augmented_tensor = get_train_transform(model_cfg)(original_image)

    # Add a batch dimension because the model expects:
    #
    #     (batch, channels, height, width)
    #
    # rather than:
    #
    #     (channels, height, width)
    model_input = augmented_tensor.unsqueeze(0).to(device)

    # Run the model without calculating gradients.
    with torch.inference_mode():
        # Calculate the raw model outputs, called logits.
        logits = model(model_input)

        # Convert the logits to probabilities.
        probabilities = torch.softmax(
            logits,
            dim=1,
        ).squeeze(0).cpu()

    # Avoid requesting more predictions than the number of available classes.
    displayed_k = min(top_k, len(classes))

    # Extract the top-k probabilities and their class indices.
    top_probabilities, top_indices = torch.topk(
        probabilities,
        k=displayed_k,
    )

    # Convert the predicted class indices to aircraft variant names.
    top_class_names = [
        classes[class_index]
        for class_index in top_indices.tolist()
    ]

    # Convert probabilities from the [0, 1] interval to percentages.
    top_percentages = top_probabilities.numpy() * 100.0

    # Store the highest-probability prediction for the return value.
    predicted_class = top_class_names[0]

    # Reuse the notebook's existing denormalize object to undo ImageNet
    # normalization before displaying the augmented tensor.
    augmented_for_display = denormalize(
        augmented_tensor
    ).clamp(0, 1)

    # Convert the denormalized tensor to a PIL image for Matplotlib.
    augmented_image = T.ToPILImage()(augmented_for_display)

    # Reverse the results so the best prediction appears at the top of the
    # horizontal bar chart.
    plot_class_names = top_class_names[::-1]
    plot_percentages = top_percentages[::-1]

    # Create a figure similar to the notebook's show_grid function, but with
    # three panels describing the complete inference process.
    fig = plt.figure(figsize=(19, 5))

    # ------------------------------------------------------------------
    # Panel 1: original dataset image
    # ------------------------------------------------------------------
    ax_original = fig.add_subplot(1, 3, 1)
    ax_original.imshow(original_image)
    ax_original.set_title(
        f"Original input\nTrue class: {classes[true_label]}",
        fontsize=11,
    )
    ax_original.axis("off")

    # ------------------------------------------------------------------
    # Panel 2: augmented image supplied to the model
    # ------------------------------------------------------------------
    ax_augmented = fig.add_subplot(1, 3, 2)
    ax_augmented.imshow(augmented_image)
    ax_augmented.set_title("Augmented model input", fontsize=11)
    ax_augmented.axis("off")

    # ------------------------------------------------------------------
    # Panel 3: model output probabilities
    # ------------------------------------------------------------------
    ax_output = fig.add_subplot(1, 3, 3)

    # Draw one horizontal bar for every displayed prediction.
    bars = ax_output.barh(
        range(displayed_k),
        plot_percentages,
        color="steelblue",
    )

    # Show the aircraft class names beside the corresponding bars.
    ax_output.set_yticks(range(displayed_k))
    ax_output.set_yticklabels(plot_class_names, fontsize=9)

    # All probabilities are percentages between zero and one hundred.
    ax_output.set_xlim(0, 100)
    ax_output.set_xlabel("Probability (%)")

    # Display the final predicted class and confidence in the panel title.
    ax_output.set_title(
        f"Prediction: {predicted_class}\n"
        f"Confidence: {top_percentages[0]:.2f}%",
        fontsize=11,
    )

    # Add a light grid to make probability values easier to compare.
    ax_output.grid(
        axis="x",
        alpha=0.25,
    )

    # Write the numerical probability beside each bar.
    for bar, percentage in zip(bars, plot_percentages):
        ax_output.text(
            min(percentage + 0.5, 97),
            bar.get_y() + bar.get_height() / 2,
            f"{percentage:.2f}%",
            va="center",
            fontsize=9,
        )

    # Prevent subplot titles and labels from overlapping.
    plt.tight_layout()

    # Display the complete visualization in the notebook.
    plt.show()

    # Return the loaded model and output values for possible later use.
    return model, probabilities, predicted_class

# Custom CNN model

This is custom low level CNN model which uses residual blocks and other techniques to improve the performance of the model. The model is designed to be trained from scratch on the FGVCAircraft dataset.


In [ ]:
class BasicBlock(nn.Module):
    """Two-convolution residual block with an optional projection shortcut."""

    def __init__(self, in_channels, out_channels, stride=1, use_batch_norm=True):
        """Construct a residual block.

        Args:
            in_channels: Number of channels in the input feature map.
            out_channels: Number of channels produced by both convolutions.
            stride: Spatial stride of the first convolution and projection shortcut.
            use_batch_norm: Whether to place BatchNorm2d after convolutions.

        Returns:
            None. The block is ready to transform tensors in ``forward``.
        """
        super().__init__()
        self.use_batch_norm = use_batch_norm

        # The first convolution may downsample and change channel width.
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=not use_batch_norm)
        if use_batch_norm:
            self.bn1 = nn.BatchNorm2d(out_channels)

        # The second convolution preserves the new spatial and channel dimensions.
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=not use_batch_norm)
        if use_batch_norm:
            self.bn2 = nn.BatchNorm2d(out_channels)

        # Project the skip path only when shape matching is required.
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            shortcut_modules = [nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=not use_batch_norm)]
            if use_batch_norm:
                shortcut_modules.append(nn.BatchNorm2d(out_channels))
            self.shortcut = nn.Sequential(*shortcut_modules)

    def forward(self, x):
        """Transform a feature map and add its identity/projection shortcut.

        Args:
            x: Input tensor shaped ``(batch, in_channels, height, width)``.

        Returns:
            ReLU-activated residual output with ``out_channels`` channels.
        """
        out = self.conv1(x)
        if self.use_batch_norm: out = self.bn1(out)
        out = F.relu(out)

        out = self.conv2(out)
        if self.use_batch_norm: out = self.bn2(out)

        # Add the skip connection before the final ReLU.
        out += self.shortcut(x)
        out = F.relu(out)
        return out


class AircraftCNN(nn.Module):
    """Custom ResNet-like aircraft classifier built only from PyTorch layers."""

    def __init__(self, num_classes: int, cfg: Dict[str, Any]):
        """Build the stem, residual stages, classifier, and initialized weights.

        Args:
            num_classes: Number of aircraft variants predicted by the final layer.
            cfg: Architecture dictionary containing channels, batch normalization,
                and classifier dropout.

        Returns:
            None. The initialized module maps RGB image batches to class logits.
        """
        super().__init__()

        self.use_batch_norm = cfg.get("use_batch_norm", True)
        channels = cfg["channels"]
        self.in_channels = channels[0]

        # The initial 7 x 7 convolution and max pool reduce image resolution by four.
        modules = [
            nn.Conv2d(3, self.in_channels, kernel_size=7, stride=2, padding=3, bias=not self.use_batch_norm)
        ]
        if self.use_batch_norm:
            modules.append(nn.BatchNorm2d(self.in_channels))
        modules.append(nn.ReLU())
        modules.append(nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        self.conv1 = nn.Sequential(*modules)

        # Each stage has two residual blocks; later stages downsample in block one.
        blocks = []
        for i, out_channels in enumerate(channels):
            stride = 1 if i == 0 else 2
            blocks.append(BasicBlock(self.in_channels, out_channels, stride=stride, use_batch_norm=self.use_batch_norm))
            blocks.append(BasicBlock(out_channels, out_channels, stride=1, use_batch_norm=self.use_batch_norm))
            self.in_channels = out_channels
        self.features = nn.Sequential(*blocks)

        # Global average pooling makes the classifier independent of feature-map size.
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Dropout(cfg.get("dropout", 0.0)),
            nn.Linear(self.in_channels, num_classes)
        )

        # Initialize trainable layers before optimization begins.
        self._init_weights()

    def _init_weights(self):
        """Initialize convolution, normalization, and linear parameters in place.

        Args:
            None.

        Returns:
            None. Parameters of this module are modified in place.
        """
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        """Map a normalized RGB image batch to aircraft-variant logits.

        Args:
            x: Tensor shaped ``(batch, 3, height, width)``.

        Returns:
            Logit tensor shaped ``(batch, num_classes)``.
        """
        x = self.conv1(x)
        x = self.features(x)
        x = self.classifier(x)
        return x


def get_custom_cnn(num_classes: int, cfg: Dict[str, Any]) -> nn.Module:
    """Create the custom CNN expected by ``run_experiment``.

    Args:
        num_classes: Number of output aircraft variants.
        cfg: Architecture and regularization configuration.

    Returns:
        An initialized ``AircraftCNN`` instance.
    """
    return AircraftCNN(num_classes, cfg)

In [ ]:
summary(AircraftCNN(100,cfg),(3,cfg["image_size"],cfg["image_size"]))

## Best model


In [ ]:
# Copy the shared best Part 1 configuration and assign its artifact name.
cfg_part1 = copy.deepcopy(cfg)
cfg_part1["run_name"] = "best_model"

# Display every setting used by the experiment.
pd.DataFrame.from_dict(cfg_part1, orient="index", columns=["value"])

### Results

In [ ]:
# Collect Part 1 results in memory for tables and comparisons.
part1_results = {}
print("Running part 1 experiments...")
SHOW_CONFUSION_MATRIX = True

# Train only when the global switch is enabled and store the result.
if RUN_TRAINING:
    part1_results[cfg_part1["run_name"]] = run_experiment(cfg_part1, get_custom_cnn)
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_part1["run_name"] + ".png"))

In [ ]:
loaded_model, probabilities, predicted_class = (
    download_model_and_show_prediction(
        model_url=drive_links["best_model"],
        dataset=data_test_preview,
        get_model_fn=get_custom_cnn,
        model_cfg=cfg_part1,
        checkpoint_path="downloaded_resnet18.pt",
        top_k=5,
    )
)

### Explanation of the results

The best custom CNN achieved **59.0% training accuracy**, **56.50% validation accuracy**, and **57.12% test accuracy**. At the selected epoch, the training loss was **2.23** and the validation loss was **1.76**, while the final test loss was **1.7122**. The similar validation and test results indicate that the model generalizes well to unseen aircraft images.

The training loss is higher than the validation and test losses because training uses augmentation, mixup, and label smoothing. These techniques make the training samples deliberately more difficult and use soft targets, whereas validation and testing use unchanged images and hard class labels. Therefore, the lower training accuracy does not necessarily indicate poor learning or underfitting.

The residual CNN can capture both low-level visual features, such as edges and textures, and higher-level aircraft structures, such as wing shapes, tails, fuselages, and engine arrangements. Residual connections also help information and gradients pass through the network, allowing it to learn deeper representations without using a predefined architecture.

The model contains only **2,824,580 trainable parameters**. Increasing the channel sizes or training for more epochs could potentially improve accuracy, but it would also increase memory usage, training time, and computational cost. In the experiments I have, the larger **11,227,812-parameter** model achieved approximately **59.47%** test accuracy—only about **2.85 percentage points** higher while using almost four times as many parameters. For this reason, the small accuracy improvement is not worth the large increase in model size and computational effort.

Overall, the selected model provides a good balance between classification accuracy, test loss, model size, and computational efficiency. It exceeds the target accuracy for Part 1 while remaining compact enough to train from scratch with limited resources.

## Part 1 ablation study

The purpose of this ablation study is to understand how each component contributes to the performance of the custom CNN. We use the best Part 1 model as the reference and change **only one setting at a time**, while keeping all other model and training settings unchanged.

After each change, we compare the training, validation, and test results with the best model. This allows us to determine whether the removed or modified component improves accuracy, reduces overfitting, or makes training more stable.

The following experiments are evaluated:

- Remove data augmentation.
- Replace light augmentation with heavy augmentation.
- Remove mixup.
- Remove label smoothing.
- Remove dropout.
- Increase dropout.
- Remove Batch Normalization.
- Decrease the learning rate.

These experiments show which design choices are important and explain why the final combination of augmentation, regularization, normalization, and optimization settings was selected for the best custom CNN.

### Remove augmentation

In [ ]:
# Disable the expensive confusion matrix for ablation runs.
SHOW_CONFUSION_MATRIX = False

# Copy the best configuration and remove geometric/photometric augmentation only.
cfg_no_aug = copy.deepcopy(cfg_part1)
cfg_no_aug["run_name"] = "best_model_no_augmentation"
cfg_no_aug["augmentation"] = "none"

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_no_aug, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_no_aug["run_name"]] = run_experiment(cfg_no_aug, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_no_aug["run_name"] + ".png"))

#### Explanation of the results

Removing data augmentation significantly reduces the model’s ability to generalize. Without augmentation, the model achieves only **11.64% best validation accuracy** and **11.41% test accuracy**, with a high test loss of **4.23**. In comparison, the best custom CNN achieves **56.50% validation accuracy**, **57.12% test accuracy**, and a test loss of **1.7122**. Removing augmentation therefore decreases test accuracy by **45.71 percentage points** and increases test loss by **2.4419**.

The main reason is overfitting. Without augmented images, training accuracy reaches **74.48%**, while validation accuracy remains close to **11.64%**. This large gap shows that the model memorizes the limited training images instead of learning features that generalize to unseen aircraft.

This problem is especially important for this dataset because it contains **100 visually similar aircraft classes** but relatively few training images for each class. The custom CNN is also trained from scratch, so it does not begin with pretrained features that are already robust to changes in object position, scale, and orientation.

Light augmentation presents the same aircraft with different crops and horizontal orientations. This encourages the CNN to focus on meaningful aircraft characteristics—such as wing shape, tail structure, fuselage proportions, and engine arrangement—instead of memorizing image backgrounds or exact aircraft positions.

Therefore, data augmentation is one of the most important components of the custom CNN. It acts as a strong regularizer, reduces overfitting, and produces the large improvement in validation and test performance.

### Make augmentation heavy

In [ ]:
# Copy the best configuration and substitute the strong augmentation policy.
cfg_strong_aug = copy.deepcopy(cfg_part1)
cfg_strong_aug["run_name"] = "best_model_strong_augmentation"
cfg_strong_aug["augmentation"] = "strong"

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_strong_aug, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_strong_aug["run_name"]] = run_experiment(cfg_strong_aug, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_strong_aug["run_name"] + ".png"))

#### Explanation of the results

Using heavy augmentation makes the custom CNN perform worse because the transformations are too aggressive for fine-grained aircraft classification. The classes in this dataset are visually similar, and the model must identify small differences in wing shape, tail design, fuselage proportions, engine position, and other structural details.

Operations such as random resized cropping can remove part of a wing, tail, or engine. Rotation can change the normal orientation of the aircraft, while stronger color and appearance changes can hide useful texture and contrast information. When several transformations are applied together, the resulting image may differ too much from a normal aircraft image.

This makes training more difficult for a CNN trained from scratch. Instead of learning the subtle features that separate similar aircraft variants, the model must spend more of its limited capacity learning invariance to heavily modified images. It can consequently underfit the actual classification task.

Heavy augmentation can also create a mismatch between training and evaluation images. The model trains on strongly transformed samples but is evaluated on normally resized and center-cropped images. If this difference becomes too large, the features learned during training transfer less effectively to validation and test images.

Light augmentation provides a better balance.

### Remove mixup

In [ ]:
# Copy the best configuration and disable mixup by both controls.
cfg_no_mixup = copy.deepcopy(cfg_part1)
cfg_no_mixup["run_name"] = "best_model_no_mixup"
cfg_no_mixup["mixup_alpha"] = 0
cfg_no_mixup["mixup_prob"] = 0

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_no_mixup, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_no_mixup["run_name"]] = run_experiment(cfg_no_mixup, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_no_mixup["run_name"] + ".png"))

#### Explanation of the results

Removing mixup makes the custom CNN more likely to memorize individual training images and become overconfident in its predictions. This is especially problematic because the model is trained from scratch on a relatively small dataset containing many visually similar aircraft classes.

Mixup creates new training samples by combining pairs of images and their labels. This exposes the model to intermediate examples and encourages it to change its predictions smoothly between classes. As a result, the decision boundaries become less sensitive to small variations in the input.

Without mixup, the CNN learns only from the original images and hard class assignments. It can create sharper and more complex decision boundaries that fit the training data well but generalize less effectively to unseen images. Training accuracy may improve because the task becomes easier, but this does not necessarily produce better validation or test performance.

Mixup also reduces overconfidence by preventing the model from assigning an absolute target to every mixed training example. This is useful for fine-grained aircraft classification, where different variants can share very similar shapes and visual characteristics.

Therefore, mixup is retained in the final custom CNN because it acts as a regularizer, encourages smoother decision boundaries, and improves generalization.

### Remove label smoothing

In [ ]:
# Copy the best configuration and set label smoothing to zero.
cfg_no_label_smoothing = copy.deepcopy(cfg_part1)
cfg_no_label_smoothing["run_name"] = "best_model_no_label_smoothing"
cfg_no_label_smoothing["label_smoothing"] = 0

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_no_label_smoothing, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_no_label_smoothing["run_name"]] = run_experiment(cfg_no_label_smoothing, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_no_label_smoothing["run_name"] + ".png"))

#### Explanation of the results

Removing label smoothing makes the custom CNN more confident about its training predictions. With ordinary one-hot labels, the correct class has a target probability of one, while every other class has a target probability of zero. This encourages the model to produce very sharp probability distributions, even when several aircraft variants are visually similar.

This behavior can cause overfitting because the model learns to treat the training labels as completely certain. It may focus too strongly on small details or noise that happen to appear in the training images rather than learning features that generalize to unseen aircraft.

Label smoothing assigns most of the target probability to the correct class while distributing a small amount across the remaining classes. This discourages extreme confidence, produces smoother decision boundaries, and makes the model less sensitive to minor variations in aircraft appearance.

This regularization is useful for fine-grained classification because many aircraft variants share similar wings, tails, engines, and fuselage structures. A moderate amount of smoothing allows the model to represent this similarity without removing the distinction between classes.

However, excessive label smoothing would make the targets too uncertain and could prevent the model from learning important differences between aircraft variants. Therefore, a small amount of label smoothing is retained in the final custom CNN as a balance between confidence and generalization.

### Remove dropout

In [ ]:
# Copy the best configuration and remove classifier dropout.
cfg_no_dropout = copy.deepcopy(cfg_part1)
cfg_no_dropout["run_name"] = "best_model_no_dropout"
cfg_no_dropout["dropout"] = 0

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_no_dropout, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_no_dropout["run_name"]] = run_experiment(cfg_no_dropout, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_no_dropout["run_name"] + ".png"))

#### Explanation of the results

Removing dropout reduces the regularization applied to the classifier. Without dropout, every feature produced by the convolutional backbone is available during every training step. This allows the classifier to depend too strongly on a small group of specific activations.

Because the custom CNN is trained from scratch on a limited number of images, this can cause the model to memorize patterns that appear only in the training set. Training accuracy may increase, but validation and test performance can become worse because these learned feature combinations may not appear in unseen images.

Dropout randomly disables part of the classifier input during training. The classifier must therefore make its prediction using different combinations of features instead of relying on only a few neurons. This encourages the network to learn more distributed and robust representations of aircraft characteristics.

This is useful for fine-grained aircraft classification because predictions should depend on several structural features—such as wings, tails, engines, and fuselage proportions—rather than one isolated visual pattern.

A moderate dropout value is therefore retained in the final custom CNN. It reduces overfitting and improves generalization without removing so much information that the model begins to underfit.

### Increase dropout

In [ ]:
# Copy the best configuration and increase classifier dropout to 0.7.
cfg_high_dropout = copy.deepcopy(cfg_part1)
cfg_high_dropout["run_name"] = "best_model_high_dropout"
cfg_high_dropout["dropout"] = 0.7

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_high_dropout, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_high_dropout["run_name"]] = run_experiment(cfg_high_dropout, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_high_dropout["run_name"] + ".png"))

#### Explanation of the results

Increasing dropout to a very high value applies excessive regularization to the classifier. With a dropout value of 0.7, approximately 70% of the classifier inputs are randomly disabled during each training step. The classifier must make predictions using only a small and constantly changing subset of the extracted features.

This can cause underfitting because the model cannot consistently combine all the visual information required to distinguish similar aircraft variants. Fine-grained classification depends on subtle combinations of features, including wing geometry, tail design, engine position, and fuselage shape. Frequently removing most of these features makes it difficult for the classifier to learn their relationships.

Very high dropout can also make optimization slower and less stable. The classifier receives a substantially different representation during every training step, so it may require more epochs to converge and can fail to use the full capacity of the convolutional backbone.

Some dropout is useful because it prevents the classifier from relying on individual features and reduces overfitting. However, too much dropout removes useful information and weakens the model. Therefore, a moderate dropout value is selected for the final custom CNN because it provides regularization without causing significant underfitting.

### Remove Batch Normalization

In [ ]:
# Copy the best configuration and disable normalization for the ablation run.
cfg_no_batch_norm = copy.deepcopy(cfg_part1)
cfg_no_batch_norm["run_name"] = "best_model_no_batch_norm"
cfg_no_batch_norm["use_batch_norm"] = False

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_no_batch_norm, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_no_batch_norm["run_name"]] = run_experiment(cfg_no_batch_norm, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_no_batch_norm["run_name"] + ".png"))

#### Explanation of the results

Removing Batch Normalization makes the custom CNN more difficult to optimize. During training, the distributions of the intermediate activations change as the convolutional weights are updated. Without normalization, these changes can become large and cause unstable gradients, slower convergence, and greater sensitivity to weight initialization and learning rate.

Batch Normalization keeps the activations of each layer within a more consistent range. This allows information and gradients to pass more reliably through the network and helps the convolutional layers learn at a stable rate.

This is particularly important in the residual blocks. The output of the convolutional path is added to the shortcut path, so their activation scales should be reasonably compatible. Without Batch Normalization, one path may become much larger than the other, reducing the effectiveness of the residual connection and making training less stable.

Batch Normalization also provides a small regularization effect because each sample is normalized using mini-batch statistics during training. Removing it eliminates this effect and can make the model more likely to overfit the limited training data.

Therefore, Batch Normalization is retained in the final custom CNN because it stabilizes the residual network, improves gradient flow, accelerates convergence, and supports better generalization.

### Decrease learning rate

In [ ]:
# Copy the best configuration and reduce the maximum OneCycle learning rate.
cfg_low_lr = copy.deepcopy(cfg_part1)
cfg_low_lr["run_name"] = "best_model_low_learning_rate"
cfg_low_lr["lr"] = 1e-4

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_low_lr, orient="index", columns=["value"])

#### Results

In [ ]:
# Run the configured ablation and store its result for the following display cell.
if RUN_TRAINING:
    part1_results[cfg_low_lr["run_name"]] = run_experiment(cfg_low_lr, get_custom_cnn)

else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part1 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part1_results))
    plot_results(part1_results, save_path=path_plots / (cfg_low_lr["run_name"] + ".png"))

#### Explanation of the results

Decreasing the learning rate makes the custom CNN learn too slowly. Because the network is trained from scratch, all convolutional filters and residual blocks must learn useful features directly from the aircraft images. A very small learning rate produces only minor parameter updates, so the model may not learn sufficiently discriminative features within the available number of epochs.

This effect is more important when using OneCycleLR. The configured learning rate is the maximum value of the schedule: training begins below this value, increases toward it, and then decreases to a value close to zero. If the maximum learning rate is already very small, most updates throughout training become extremely small.

As a result, the first layers may learn basic edges and textures, but the deeper layers may not fully learn the subtle combinations of wing shape, tail structure, fuselage proportions, and engine arrangement required to distinguish similar aircraft variants. This leads to underfitting rather than unstable training.

A lower learning rate could potentially work if the model were trained for substantially more epochs, but this would increase computational cost and training time. Therefore, the higher learning rate is retained because it allows the CNN to converge more effectively within the selected training budget.

## Complete list of experiments and their results

### My model-development journey

I started by changing the augmentation policy. The original strong augmentation used aggressive cropping and color changes that could remove or distort important aircraft details. I made the crop range narrower, reduced the color jitter, added only a small rotation, and made random erasing less aggressive. My intention was to preserve the wings, tail, engines, and fuselage while still introducing enough variation to reduce overfitting.

My first architecture was a plain VGG-style CNN constructed from convolutional blocks. Although changing the augmentation made the input images more suitable, the first model still performed poorly. This showed me that improving augmentation alone was not enough; the architecture also needed to extract stronger features.

I then created `cnn_ne_config`. I used a 5×5 kernel in the first layer, replaced ReLU with SiLU, and added a larger classifier containing a hidden linear layer. This substantially improved the result, but the model still struggled to distinguish the 100 visually similar aircraft variants.

After discussing the model with the teaching assistant, I replaced the plain convolutional blocks with residual blocks. The first residual model, `residual_block_1`, performed better because skip connections improved gradient flow and allowed the network to learn deeper representations.

Next, I changed the learning-rate scheduler to cosine annealing in `residual_block_2_lr`. I also experimented with early stopping based on validation loss while continuing to save the model according to validation accuracy. This did not improve the model as expected. The scheduler reduced the learning rate too smoothly, and validation loss was not fully aligned with the accuracy metric used to select the best classifier.

I therefore returned to OneCycleLR, changed early stopping back to validation accuracy, and stopped concatenating the validation data with the training data. This produced `residual_block_lr_onecycle_old` and improved the result.

I then reduced the channels from `[64, 128, 256, 512]` to `[32, 64, 128, 256]`, removed block dropout, disabled mixup for that stage, changed weight decay to `1e-4`, and added explicit weight initialization. This created `residual_block_lr_onecycle`. Its accuracy decreased, but the model became almost four times smaller, which made it much more computationally efficient.

The next major improvement came from the data pipeline. I compared light and strong augmentation together with different validation-cropping choices. Light training augmentation with center-cropped validation images produced the best result and allowed the custom CNN to exceed the target accuracy. This became the reference configuration for the hyperparameter experiments.

### Architecture and pipeline development

| Run | Main change | Best validation | Test accuracy | Test loss | Best epoch | Parameters |
| --- | --- | ---: | ---: | ---: | ---: | ---: |
| `custom_cnn_vgg_style_mixup` | Initial VGG-style CNN | 8.49% | 8.73% | 3.6522 | 26 | 2,379,588 |
| `cnn_mixup_balanced_check` | Short training-pipeline sanity check | 1.17% | 1.17% | 4.6172 | 2 | 4,738,596 |
| `cnn_ne_config` | 5×5 first kernel, SiLU and larger classifier | 27.30% | 27.48% | 2.8004 | 27 | 5,005,348 |
| `part1_custom_cnn_best` | Earlier custom-CNN checkpoint | 29.91% | 28.89% | 2.8953 | 31 | 11,227,812 |
| `residual_block_1` | Introduced residual blocks | 34.41% | 34.26% | 2.5517 | 25 | 11,227,812 |
| `residual_block_2_lr` | Cosine annealing and validation-loss early stopping | 32.79% | 33.45% | 2.7625 | 48 | 11,227,812 |
| `residual_block_2_lr_80epochs` | Longer cosine-scheduler experiment | 35.28% | 36.48% | 2.7968 | 47 | 11,227,812 |
| `residual_block_lr_onecycle_old` | Returned to OneCycleLR and accuracy-based stopping | 38.70% | 38.91% | 2.7290 | 70 | 11,227,812 |
| `residual_block_lr_onecycle` | Reduced channels and regularization | 33.21% | 33.87% | 2.7431 | 42 | 2,824,580 |
| `residual_block_aug_strong_nocrop` | Strong augmentation without center crop | 34.86% | 33.36% | 2.9013 | 41 | 11,227,812 |
| `residual_block_aug_both` | Combined augmentation/cropping experiment | 21.00% | 21.00% | 3.5979 | 45 | 11,227,812 |
| `residual_block_aug_light` | Light augmentation with the selected evaluation pipeline | 54.46% | 54.91% | 1.7650 | 46 | 11,227,812 |

### Hyperparameter tuning

After finding a stable architecture and data pipeline, I tuned each important hyperparameter separately. I changed one main setting at a time so that I could understand its effect on convergence, regularization, overfitting, and generalization.

The best value from an individual experiment was not always the best value when combined with the other settings. For example, strong mixup and high dropout worked well separately, but combining them applied too much regularization and made the final model worse. I therefore selected the final configuration based on the complete training behavior rather than simply combining every individually strong value.

| Experiments | Setting investigated | Observation | Final decision |
| --- | --- | --- | --- |
| `base_batch_norm_false` | Batch Normalization | Removing normalization made optimization less stable and reduced generalization. | Keep Batch Normalization enabled. |
| `base_droput_0.1`, `base_droput_0.3`, `base_froput_0.5` | Dropout | Higher dropout provided strong standalone regularization, but it became excessive when combined with mixup and label smoothing. | Use lower dropout in the final combined model. |
| `base_label_smoothing_0.0`, `base_label_smoothing_0.1`, `base_label_smoothing_0.3` | Label smoothing | No smoothing increased confidence and overfitting, while excessive smoothing weakened class distinctions and increased loss. | Use moderate label smoothing. |
| `base_lr_1e-4`, `base_lr_3e-4`, `base_lr_3e-3` | Learning rate | Small learning rates made the model converge too slowly under OneCycleLR. The larger tested rate used the training budget more effectively. | Use `3e-3`. |
| `base_mixup_alpha_0`, `base_mixup_alpha_0.4`, `base_mixup_alpha_1` | Mixup strength | Removing mixup reduced regularization. Very strong mixup worked well alone but became too aggressive when combined with dropout. | Use a strong but slightly reduced mixup alpha. |
| `base_mixup_prob_0.3`, `base_mixup_prob_0.5`, `base_mixup_prob_0.5_70epoch`, `base_mixup_prob_0.7`, `base_mixup_prob_1` | Mixup probability | Applying mixup too rarely reduced its benefit, while applying it to every batch gave the model too few unchanged examples. | Use mixup frequently, but not for every batch. |
| `base_no_aug` | Data augmentation | Removing augmentation caused severe overfitting because the model repeatedly observed the same limited training images. | Keep light augmentation. |
| `base_no_random` | Random erasing | Random erasing had a smaller effect than the main augmentation policy. Aggressive erasing could remove important aircraft details. | Keep only mild random erasing. |
| `base_reduce_channel` | Model capacity | Reducing the channel widths decreased computational cost and parameter count while preserving sufficient classification performance. | Select the compact architecture. |
| `base_wd_1e-5`, `base_wd_5e-4`, `base_wd_1e-3` | Weight decay | Weak weight decay provided limited regularization, while excessive weight decay could restrict learning. | Use `5e-4` as a balanced value. |

### Training-history visualization

The following cell displays the combined Part 1 experiment history using the previously defined image-download function:



In [ ]:
download_and_show_image("base_build")


### Selecting the final Part 1 model

The hyperparameter experiments showed that light augmentation was essential and that Batch Normalization should remain enabled. A sufficiently large OneCycleLR maximum learning rate was necessary for convergence. Moderate label smoothing improved generalization, while mixup and dropout both provided useful regularization.

However, selecting every individually strong regularization setting and combining them did not produce the best model. Strong mixup and high dropout together removed too much reliable information during training. I therefore reduced dropout, used a slightly less aggressive mixup configuration, trained for 70 epochs, and retained the stronger learning rate.

The final configuration was:

| Setting | Final value |
| --- | --- |
| Channels | `[32, 64, 128, 256]` |
| Augmentation | Light |
| Random erasing | Enabled |
| Dropout | 0.15 |
| Mixup alpha | 0.85 |
| Mixup probability | 0.75 |
| Label smoothing | 0.1 |
| Learning rate | `3e-3` |
| Weight decay | `5e-4` |
| Epochs | 70 |
| Trainable parameters | 2,824,580 |
| Best validation accuracy | 56.05% |
| Test accuracy | 56.62% |
| Test loss | 1.7168 |

### Evidence for my final choices

- Light augmentation reduced overfitting while preserving the fine details needed to distinguish aircraft variants.
- Residual blocks improved gradient flow and learned stronger representations than the plain VGG-style model.
- OneCycleLR worked better for this training budget than cosine annealing.
- Validation-accuracy early stopping was better aligned with the model-selection objective.
- Batch Normalization stabilized the residual blocks and made optimization more reliable.
- Moderate label smoothing reduced overconfidence without making the class targets too uncertain.
- Mixup improved generalization, but combining very strong mixup with high dropout caused excessive regularization.
- A higher learning rate allowed the network trained from scratch to converge within the available epochs.
- Reducing the channel width substantially decreased the parameter count and computational cost.

The wider model could produce somewhat higher accuracy and lower loss, and additional epochs could potentially improve it further. However, the strongest recorded wider result improved test accuracy by only about **2.85 percentage points** while increasing the parameter count from **2,824,580** to **11,227,812**—almost four times as many parameters.

From my perspective, this accuracy improvement was not worth the much larger model, additional memory usage, and longer training time. I therefore selected the compact 256-channel model as my final Part 1 model and carried its best training hyperparameters forward to the ResNet-18 experiments in Part 2.

# Part 2

This part uses the PyTorch ResNet-18 model with ImageNet-1K V1 weights


## Load pretrained ResNet-18 model

In [ ]:
def set_requires_grad(layer: torch.nn.Module, train: bool) -> None:
    """Enable or disable gradient computation for every parameter in a layer.

    Args:
        layer: PyTorch module whose parameters will be frozen or unfrozen.
        train: True to train the layer; False to freeze it.

    Returns:
        None. Each parameter's ``requires_grad`` flag is modified in place.
    """
    for p in layer.parameters():
        p.requires_grad = train


def get_resnet18_model(num_classes: int, cfg: Dict[str, Any]) -> nn.Module:
    """Create ImageNet-1K V1 ResNet-18 with a task-specific classifier.

    Args:
        num_classes: Number of FGVC-Aircraft variants predicted by the new head.
        cfg: Configuration containing classifier ``dropout`` and
            ``freeze_backbone``.

    Returns:
        A pretrained ResNet-18 whose final layer outputs ``num_classes`` logits.
        When requested, all convolutional backbone stages are frozen.
    """
    # Load the assignment-required ImageNet-1K V1 pretrained weights.
    weights = ResNet18_Weights.IMAGENET1K_V1
    model = resnet18(weights=weights)

    # Replace the 1000-class ImageNet head with dropout and a 100-class linear layer.
    classifier = nn.Sequential(
        nn.Dropout(cfg["dropout"]),
        nn.Linear(model.fc.in_features, num_classes)
    )
    model.fc = classifier

    # Freezing leaves only the new classifier parameters trainable.
    if cfg["freeze_backbone"]:
        set_requires_grad(model.conv1, False)
        set_requires_grad(model.bn1, False)
        set_requires_grad(model.layer1, False)
        set_requires_grad(model.layer2, False)
        set_requires_grad(model.layer3, False)
        set_requires_grad(model.layer4, False)
    else:
        set_requires_grad(model.conv1, True)
        set_requires_grad(model.bn1, True)
        set_requires_grad(model.layer1, True)
        set_requires_grad(model.layer2, True)
        set_requires_grad(model.layer3, True)
        set_requires_grad(model.layer4, True)
        

    return model

In [ ]:
summary(get_resnet18_model(100,cfg),(3,cfg["image_size"],cfg["image_size"]))

## Part 2A: ResNet-18 with Part 1 hyperparameters

The training hyperparameters are copied from the best model from part 1. The only necessary architecture change is replacing the final classifier of ResNet-18.
I will try both freezing the backbone and training the whole model, and compare the results.


### Freeze backbone

In [ ]:
# Reuse every Part 1 training hyperparameter, then freeze the ResNet backbone.
cfg_resnet_same_freeze = copy.deepcopy(cfg_part1)
cfg_resnet_same_freeze["run_name"] = "resnet18_same_hyperparameters_freeze_backbone"
cfg_resnet_same_freeze["dropout"] = 0.0
cfg_resnet_same_freeze["freeze_backbone"] = True
cfg_resnet_same_freeze["head_lr"] = None

# Display the exact Part 2A frozen-backbone configuration.
pd.DataFrame.from_dict(cfg_resnet_same_freeze, orient="index", columns=["value"])

### Result

In [ ]:
# Keep Part 2 results for subsequent comparison with whole-model training.
part2_results = {}

# Train the frozen-backbone model only when globally enabled.
if RUN_TRAINING:
    part2_results[cfg_resnet_same_freeze["run_name"]] = run_experiment(cfg_resnet_same_freeze, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_same_freeze["run_name"] + ".png"))

### Explanation of the results

Freezing the ResNet-18 backbone produces worse performance because only the new classifier can learn from the aircraft dataset. The convolutional layers retain the features learned from ImageNet and cannot adapt to this fine-grained classification task.

ImageNet pretraining provides useful general features such as edges, textures, and object shapes. However, distinguishing aircraft variants requires more specialized features, including small differences in wing geometry, tail design, engine position, windows, and fuselage proportions. A newly initialized linear classifier cannot recover information that the frozen backbone does not represent clearly.

The Part 1 hyperparameters were selected for training the complete custom CNN from scratch. In that model, augmentation, mixup, label smoothing, and the learning-rate schedule affect both the feature extractor and the classifier. When the ResNet backbone is frozen, these settings regularize only a small classifier and may make its learning problem unnecessarily difficult.

Freezing the backbone also prevents the deeper residual layers from shifting their attention from general ImageNet objects to aircraft-specific structures. The classifier can only rearrange the fixed features it receives, which creates a performance limit even if training continues for more epochs.

Therefore, freezing the backbone is computationally efficient, but it is not suitable for achieving the best performance on this dataset. Fine-tuning the complete ResNet-18 allows both the pretrained features and the classifier to adapt to the subtle differences among aircraft variants.

### Train the whole model

In [ ]:
# Reuse the Part 1 hyperparameters but allow every ResNet layer to update.
cfg_resnet_same_whole = copy.deepcopy(cfg_part1)
cfg_resnet_same_whole["run_name"] = "resnet18_same_hyperparameters_whole_model"
cfg_resnet_same_whole["dropout"] = 0.0
cfg_resnet_same_whole["freeze_backbone"] = False
cfg_resnet_same_whole["early_stop_patience"] = 20
cfg_resnet_same_whole["head_lr"] = None

# Display the exact Part 2A whole-model configuration.
pd.DataFrame.from_dict(cfg_resnet_same_whole, orient="index", columns=["value"])

### Result

In [ ]:
# Train the whole ResNet model and append it to the in-memory comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_same_whole["run_name"]] = run_experiment(cfg_resnet_same_whole, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_same_whole["run_name"] + ".png"))

### Explanation of the results

Training all **11,227,812** parameters with the same Part 1 hyperparameters reaches **71.95%** best validation accuracy at epoch **65**, **72.13%** test accuracy, and **1.3300** test loss. Compared with freezing the backbone, full fine-tuning gains **37.62 percentage points** of test accuracy and reduces test loss by **1.3733**. This is the cleanest controlled comparison in the notebook because the two Part 2A configurations differ primarily in whether the backbone receives gradients (apart from the recorded patience change).

## Part 2B: full fine-tuning

After the  baseline, we will fine-tune all the hyper parmas


In [ ]:
# Start from the frozen Part 2A configuration and set the current Part 2B values.
# The preserved output was generated by an older configuration documented below.
cfg_resnet_best = copy.deepcopy(cfg_resnet_same_freeze)
cfg_resnet_best["run_name"] = "resnet18_best"
cfg_resnet_best["augmentation"] = "light"
cfg_resnet_best["num_epochs"] = 70
cfg_resnet_best["lr"] = 9e-4
cfg_resnet_best["wd"] = 5e-4
cfg_resnet_best["head_lr"] = 3e-3
cfg_resnet_best["dropout"] = 0
cfg_resnet_best["mixup_alpha"] = 1
cfg_resnet_best["mixup_prob"] = 0.5
cfg_resnet_best["label_smoothing"] = 0.10
cfg_resnet_best["freeze_backbone"] = False

# Display the current source configuration.
pd.DataFrame.from_dict(cfg_resnet_best, orient="index", columns=["value"])

### Results

In [ ]:
# Train the current Part 2B configuration and add it to the in-memory comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_best["run_name"]] = run_experiment(cfg_resnet_best, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_best["run_name"] + ".png"))

#### Explanation of the results

The preserved `resnet18_best` output reports **68.02%** best validation accuracy at epoch **64**, **68.86%** test accuracy, and **1.5268** test loss. The saved summary records **11,227,812 trainable parameters** and the history contains all 70 epochs.

There is a reproducibility mismatch that must be stated explicitly: `outputs/summaries/resnet18_best_summary.json` records the run that produced this output with backbone learning rate **2e-4**, head learning rate **1e-3**, mixup alpha **0.9**, and mixup probability **0.3**. The current source cell above shows backbone learning rate `3e-4` and mixup disabled. Therefore the preserved numerical output is explained using the saved JSON configuration; rerunning the current cell would be a different experiment. The recorded 68.86% result is also below the **72.13%** Part 2A whole-model result.

## Part 2B ablation study

The purpose of this ablation study is to understand how each fine-tuning choice contributes to the performance of ResNet-18. We use the best Part 2 configuration as the reference and change **only one setting at a time**, while keeping the model, training procedure, and all other hyperparameters unchanged.

After each change, we compare the training, validation, and test behavior with the best model. This allows us to determine whether each component helps the pretrained network adapt to the fine-grained differences among aircraft variants.

The following experiments are evaluated:

- Freeze the pretrained backbone.
- Remove data augmentation.
- Replace light augmentation with stronger augmentation.
- Remove mixup.
- Remove label smoothing.
- Increase classifier dropout.
- Use different learning rates for the backbone and classifier head.

These experiments show which choices are important for transfer learning and explain why the final combination of full fine-tuning, augmentation, regularization, and optimization settings was selected for ResNet-18.

<a id="Part-2B-freeze-the-backbone"></a>

### Freeze the backbone

In [ ]:
# Copy the Part 2B configuration and freeze every pretrained convolutional stage.
cfg_resnet_freeze = copy.deepcopy(cfg_resnet_best)
# Restore the saved-result reference values before changing the ablated factor.
cfg_resnet_freeze["lr"] = 2e-4
cfg_resnet_freeze["head_lr"] = 1e-3
cfg_resnet_freeze["mixup_alpha"] = 0.9
cfg_resnet_freeze["mixup_prob"] = 0.3
cfg_resnet_freeze["run_name"] = "resnet18_best_freeze_backbone"
cfg_resnet_freeze["freeze_backbone"] = True
cfg_resnet_freeze["head_lr"] = None

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_resnet_freeze, orient="index", columns=["value"])

#### Results

In [ ]:
# Run this single-factor ResNet ablation and append it to the Part 2 comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_freeze["run_name"]] = run_experiment(cfg_resnet_freeze, get_resnet18_model)
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_freeze["run_name"] + ".png"))

#### Explanation of the results

Freezing the backbone prevents the pretrained convolutional layers from adapting to the aircraft dataset. Only the newly initialized classifier is trained, so it must separate all aircraft variants using fixed features learned from general ImageNet objects.

These pretrained features are useful for recognizing basic edges, textures, and shapes, but fine-grained aircraft classification requires more specialized information. The network must identify subtle differences in wing geometry, tail structure, engine arrangement, windows, and fuselage proportions. If the backbone is frozen, its deeper layers cannot reorganize their features to emphasize these details.

The data augmentation and regularization settings also become less effective because they can only influence the classifier. The feature extractor cannot learn to produce consistent aircraft-specific representations from the augmented images.

Freezing the backbone reduces training time and computational cost, but it creates a strong performance limit. Therefore, the complete ResNet-18 is fine-tuned in the final model so that both the pretrained feature extractor and the classifier can adapt to the aircraft variants.

<a id="Part-2B-remove-augmentation"></a>

### Remove augmentation

In [ ]:
# Copy the Part 2B configuration and replace training augmentation with resizing only.
cfg_resnet_no_aug = copy.deepcopy(cfg_resnet_best)
# Restore the saved-result reference values before changing the ablated factor.
cfg_resnet_no_aug["lr"] = 2e-4
cfg_resnet_no_aug["head_lr"] = 1e-3
cfg_resnet_no_aug["mixup_alpha"] = 0.9
cfg_resnet_no_aug["mixup_prob"] = 0.3
cfg_resnet_no_aug["run_name"] = "resnet18_best_no_augmentation"
cfg_resnet_no_aug["augmentation"] = "none"

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_resnet_no_aug, orient="index", columns=["value"])

#### Results

In [ ]:
# Run this single-factor ResNet ablation and append it to the Part 2 comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_no_aug["run_name"]] = run_experiment(cfg_resnet_no_aug, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_no_aug["run_name"] + ".png"))

#### Explanation of the results

Removing data augmentation makes the fine-tuned ResNet-18 more likely to overfit the limited training images. Because the complete network is trainable, it can adapt very closely to the exact aircraft positions, image backgrounds, scales, and orientations present in the training set.

This is particularly problematic for fine-grained aircraft classification. The model should learn stable structural characteristics—such as wing geometry, tail design, engine placement, and fuselage proportions—rather than memorizing the composition of individual images.

Light augmentation shows the network slightly different versions of the same aircraft through random crops and horizontal flips. This teaches the pretrained features to remain useful when the aircraft position or visible region changes and prevents the backbone from becoming too specialized to the training images.

Without augmentation, training may become easier and training accuracy may increase, but the learned features generalize less effectively to validation and test images. Therefore, light data augmentation is retained because it reduces overfitting while preserving the fine details required to distinguish aircraft variants.

<a id="Part-2B-make-augmentation-heavy"></a>

### Make augmentation heavy

In [ ]:
# Copy the Part 2B configuration and use the ResNet-specific strong policy.
cfg_resnet_strong_aug = copy.deepcopy(cfg_resnet_best)
# Restore the saved-result reference values before changing the ablated factor.
cfg_resnet_strong_aug["lr"] = 2e-4
cfg_resnet_strong_aug["head_lr"] = 1e-3
cfg_resnet_strong_aug["mixup_alpha"] = 0.9
cfg_resnet_strong_aug["mixup_prob"] = 0.3
cfg_resnet_strong_aug["run_name"] = "resnet18_best_strong_augmentation"
cfg_resnet_strong_aug["augmentation"] = "resnet_strong"

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_resnet_strong_aug, orient="index", columns=["value"])

#### Results

In [ ]:
# Run this single-factor ResNet ablation and append it to the Part 2 comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_strong_aug["run_name"]] = run_experiment(cfg_resnet_strong_aug, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_strong_aug["run_name"] + ".png"))

#### Explanation of the results

Using strong augmentation makes the ResNet-18 fine-tuning task unnecessarily difficult. Aggressive cropping can remove or distort important aircraft components, while stronger geometric and appearance transformations can change the small visual details needed to separate similar variants.

ResNet-18 begins with useful ImageNet features, but full fine-tuning changes these features according to the new training images. If the augmented images are too distorted, the backbone may learn to ignore precisely the details required for classification, such as wing shape, tail geometry, engine position, windows, and fuselage proportions.

Strong augmentation can also create a mismatch between training and evaluation. The network trains on heavily transformed aircraft but is evaluated on normally resized and center-cropped images. If this difference becomes too large, the learned representation transfers less effectively to validation and test samples.

Light augmentation provides a better balance. It introduces changes in crop and orientation that reduce overfitting while preserving the aircraft’s structure and fine-grained characteristics. Therefore, light augmentation is selected for the final ResNet-18 model instead of the stronger policy.

<a id="Part-2B-remove-mixup"></a>

### Remove mixup

In [ ]:
# Copy the Part 2B configuration and disable mixup by both controls.
cfg_resnet_no_mixup = copy.deepcopy(cfg_resnet_best)
# Restore the saved-result reference values before changing the ablated factor.
cfg_resnet_no_mixup["lr"] = 2e-4
cfg_resnet_no_mixup["head_lr"] = 1e-3
cfg_resnet_no_mixup["mixup_alpha"] = 0.9
cfg_resnet_no_mixup["mixup_prob"] = 0.3
cfg_resnet_no_mixup["run_name"] = "resnet18_best_no_mixup"
cfg_resnet_no_mixup["mixup_alpha"] = 0
cfg_resnet_no_mixup["mixup_prob"] = 0

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_resnet_no_mixup, orient="index", columns=["value"])

#### Results

In [ ]:
# Run this single-factor ResNet ablation and append it to the Part 2 comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_no_mixup["run_name"]] = run_experiment(cfg_resnet_no_mixup, get_resnet18_model)
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_no_mixup["run_name"] + ".png"))

#### Explanation of the results

No exact zero-mixup ResNet artifact exists in `summary_resnet.csv`, so an ablation result is not invented. The closest saved frozen runs are `resnet_mixup_prob_0.3` (**29.43%** test), `resnet_mixup_alpha_0.4` (**27.87%**), and `resnet_base` with alpha/probability 1 (**25.41%**). They show sensitivity to mixup strength, but none answers the zero-mixup question.

<a id="Part-2B-remove-label-smoothing"></a>

### Remove label smoothing

In [ ]:
# Copy the Part 2B configuration and set label smoothing to zero.
cfg_resnet_no_label_smoothing = copy.deepcopy(cfg_resnet_best)
# Restore the saved-result reference values before changing the ablated factor.
cfg_resnet_no_label_smoothing["lr"] = 2e-4
cfg_resnet_no_label_smoothing["head_lr"] = 1e-3
cfg_resnet_no_label_smoothing["mixup_alpha"] = 0.9
cfg_resnet_no_label_smoothing["mixup_prob"] = 0.3
cfg_resnet_no_label_smoothing["run_name"] = "resnet18_best_no_label_smoothing"
cfg_resnet_no_label_smoothing["label_smoothing"] = 0

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_resnet_no_label_smoothing, orient="index", columns=["value"])

#### Results

In [ ]:
# Run this single-factor ResNet ablation and append it to the Part 2 comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_no_label_smoothing["run_name"]] = run_experiment(cfg_resnet_no_label_smoothing, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_no_label_smoothing["run_name"] + ".png"))

#### Explanation of the results

Removing label smoothing forces the fine-tuned ResNet-18 to learn completely certain one-hot targets. This encourages the classifier to assign nearly all probability to one class, even when several aircraft variants have very similar visual characteristics.

Because the complete network is fine-tuned, this overconfidence can also affect the backbone. The deeper layers may adapt too strongly to small details or noise in the training images, producing sharper decision boundaries that do not generalize well to unseen aircraft.

A moderate amount of label smoothing prevents extreme confidence by assigning most of the target probability to the correct class and a small amount to the other classes. This encourages smoother decision boundaries and helps preserve the general representations learned during ImageNet pretraining.

This is useful for aircraft classification because related variants may share similar wings, tails, engines, and fuselage structures. However, excessive smoothing could hide meaningful differences between classes and cause underfitting.

Therefore, moderate label smoothing is retained in the final ResNet-18 configuration to reduce overconfidence and improve generalization without weakening the fine-grained class distinctions.

<a id="Part-2B-increase-dropout"></a>

### Increase dropout

In [ ]:
# Copy the Part 2B configuration and add 0.3 dropout before the classifier.
cfg_resnet_high_dropout = copy.deepcopy(cfg_resnet_best)
# Restore the saved-result reference values before changing the ablated factor.
cfg_resnet_high_dropout["lr"] = 2e-4
cfg_resnet_high_dropout["head_lr"] = 1e-3
cfg_resnet_high_dropout["mixup_alpha"] = 0.9
cfg_resnet_high_dropout["mixup_prob"] = 0.3
cfg_resnet_high_dropout["run_name"] = "resnet18_best_dropout_0.3"
cfg_resnet_high_dropout["dropout"] = 0.3

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_resnet_high_dropout, orient="index", columns=["value"])

#### Results

In [ ]:
# Run this single-factor ResNet ablation and append it to the Part 2 comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_high_dropout["run_name"]] = run_experiment(cfg_resnet_high_dropout, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_high_dropout["run_name"] + ".png"))

#### Explanation of the results

Increasing classifier dropout adds additional regularization by randomly removing part of the feature vector before the final prediction layer. This can be useful when the classifier is overfitting, but it can also remove important information learned by the pretrained backbone.

The final ResNet feature vector contains a compact representation of the aircraft image. Fine-grained classification depends on combining several subtle features, such as wing shape, tail design, engine position, and fuselage proportions. Increasing dropout prevents the classifier from using many of these features together during each training step.

The ResNet-18 model is already regularized through data augmentation, mixup, label smoothing, weight decay, and ImageNet pretraining. Adding stronger dropout on top of these techniques can make the total regularization excessive. This may slow classifier learning and cause underfitting rather than improve generalization.

Therefore, classifier dropout should remain low or disabled unless the training curves show clear classifier overfitting. For the final ResNet-18 configuration, the existing regularization methods are sufficient, so additional dropout is not necessary.

<a id="Part-2B-differential-learning-rates"></a>

### Use differential backbone and head learning rates

In [ ]:
# Copy the Part 2B configuration and use the strongest saved differential rates.
cfg_resnet_differential_lr = copy.deepcopy(cfg_resnet_best)
# Restore the saved-result reference values before changing the ablated factor.
cfg_resnet_differential_lr["lr"] = 3e-5
cfg_resnet_differential_lr["head_lr"] = 1e-4
cfg_resnet_differential_lr["mixup_alpha"] = 0.9
cfg_resnet_differential_lr["mixup_prob"] = 0.3
cfg_resnet_differential_lr["run_name"] = "resnet18_best_backbone_1e-4_head_1e-3"
cfg_resnet_differential_lr["lr"] = 3e-5
cfg_resnet_differential_lr["head_lr"] = 3e-4
cfg_resnet_differential_lr["freeze_backbone"] = False

# Display the exact ablation configuration.
pd.DataFrame.from_dict(cfg_resnet_differential_lr, orient="index", columns=["value"])

#### Results

In [ ]:
# Run this single-factor ResNet ablation and append it to the Part 2 comparison.
if RUN_TRAINING:
    part2_results[cfg_resnet_differential_lr["run_name"]] = run_experiment(cfg_resnet_differential_lr, get_resnet18_model)
    
else:
    print("RUN_TRAINING is False")

In [ ]:
# Display the current part2 results table and save its accuracy curves.
if RUN_TRAINING:
    display(results_to_table(part2_results))
    plot_results(part2_results, save_path=path_plots / (cfg_resnet_differential_lr["run_name"] + ".png"))

#### Explanation of the results

Using different learning rates for the backbone and classifier head is appropriate because these parts of the network begin training from different states.

The ResNet-18 backbone already contains useful features learned from ImageNet. It should be updated with a smaller learning rate so that it can gradually adapt to aircraft-specific structures without destroying the general visual features acquired during pretraining. If the backbone learning rate is too large, the pretrained representation can change too quickly, producing unstable training or catastrophic forgetting.

In contrast, the classifier head is newly initialized and does not contain useful pretrained information. It requires a larger learning rate so that it can quickly learn how to map the backbone features to the aircraft classes. If the head uses the same small learning rate as the backbone, it may learn too slowly within the available training epochs.

However, the backbone learning rate must not be so small that the feature extractor remains almost unchanged. Fine-grained aircraft recognition requires the deeper layers to adapt to details such as wing geometry, tail structure, engine arrangement, and fuselage proportions.

Therefore, the final fine-tuning strategy uses a smaller learning rate for the pretrained backbone and a larger learning rate for the new classifier. This balances preservation of the pretrained features with sufficient adaptation to the aircraft dataset.

## Final comparison

The final table compares the custom CNN, the ablations, and the ResNet-18 experiments.


In [ ]:
# Merge every in-memory Part 1 and Part 2 result into one final comparison.
all_results = {}
all_results.update(part1_results)
all_results.update(part2_results)

# Persist and display the table only when at least one experiment has run.
if len(all_results) > 0:
    table_all = results_to_table(all_results)
    table_all.to_csv(path_outputs / "final_model_comparison.csv", index=False)
    display(table_all)
    plot_results(all_results)
else:
    print("Run the training cells to populate the final comparison table.")

### Result of ResNet training

This report is generated from `outputs/analysis/summary_resnet.csv`, the referenced histories/summaries, and the preserved notebook outputs. The analysis CSV contains two distinct regimes, so they are not mixed into a false controlled ranking.

![ResNet experiment histories](https://drive.google.com/file/d/1r-C6iwn_pY-pWrcgMse1nj8Gku9Z6KEM/view?usp=sharing)

#### Frozen-backbone historical experiments

Only **51,300** classifier parameters are trainable in these runs.

| Run | Best val | Test | Loss | Epoch | Trainable | Key setting |
| --- | --- | --- | --- | --- | --- | --- |
| resnet_aug_light | 29.34% | 29.07% | 3.3025 | 40 | 51,300 | aug=light; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet_base | 23.64% | 25.41% | 3.4751 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet_droput_0.3 | 24.93% | 26.49% | 3.3337 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.3; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet_label_smoothing_0.1 | 23.34% | 25.05% | 3.4104 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.1; mixup=(1.0, 1.0); erase=True |
| resnet_label_smoothing_0.25 | 23.61% | 25.41% | 3.5127 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.25; mixup=(1.0, 1.0); erase=True |
| resnet_mixup_alpha_0.4 | 26.07% | 27.87% | 3.3346 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(0.4, 1.0); erase=True |
| resnet_mixup_prob_0.3 | 27.72% | 29.43% | 3.2198 | 37 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 0.3); erase=True |
| resnet_mixup_prob_0.7 | 25.23% | 27.42% | 3.3663 | 37 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 0.7); erase=True |
| resnet_no_aug | 19.83% | 19.89% | 3.5792 | 39 | 51,300 | aug=none; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet_no_random | 24.03% | 25.32% | 3.4707 | 38 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=False |
| resnet_wd_1e-3 | 23.64% | 25.41% | 3.4753 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet_wd_1e-5 | 23.64% | 25.41% | 3.4751 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet_wd_5e-4 | 23.64% | 25.41% | 3.4752 | 36 | 51,300 | aug=resnet_strong; lr/head=0.001/; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |

The frozen baseline `resnet_base` records **25.41%** test accuracy. Light augmentation is the strongest saved frozen variant at **29.07%**, closely followed by mixup probability 0.3 at **29.43%**. Removing augmentation falls to **19.89%**. Weight-decay values `1e-5`, `5e-4`, and `1e-3` all reproduce **25.41%** test accuracy to the displayed precision, so this sweep does not show a useful weight-decay effect while the backbone is frozen.

#### Historical full-fine-tuning learning-rate sweep

All **11,227,812** parameters are trainable in these runs, with separate backbone/head learning rates.

| Run | Best val | Test | Loss | Epoch | Trainable | Key setting |
| --- | --- | --- | --- | --- | --- | --- |
| resnet18_backbone_1e-4_head_1e-3 | 61.57% | 61.93% | 1.6462 | 38 | 11,227,812 | aug=resnet_strong; lr/head=0.0001/0.001; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet18_backbone_1e-4_head_5e-4 | 60.46% | 61.81% | 1.6844 | 40 | 11,227,812 | aug=resnet_strong; lr/head=0.0001/0.0005; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet18_backbone_1e-5_head_1e-4 | 27.93% | 28.98% | 3.1802 | 38 | 11,227,812 | aug=resnet_strong; lr/head=1e-05/0.0001; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet18_backbone_3e-5_head_3e-4 | 49.08% | 50.14% | 2.1719 | 39 | 11,227,812 | aug=resnet_strong; lr/head=3e-05/0.0003; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |
| resnet18_backbone_5e-5_head_5e-4 | 55.63% | 56.83% | 1.8871 | 35 | 11,227,812 | aug=resnet_strong; lr/head=5e-05/0.0005; dropout=0.5; LS=0.2; mixup=(1.0, 1.0); erase=True |

The sweep improves monotonically across the tested pairs up to backbone `1e-4` and head `1e-3`, which reaches **61.57%** best validation accuracy and **61.93%** test accuracy. The nearby `1e-4/5e-4` pair is almost tied at **61.81%** test accuracy.

#### Final notebook runs

| Run | Best validation | Test accuracy | Test loss | Best epoch | Trainable parameters |
| --- | --- | --- | --- | --- | --- |
| `resnet18_same_hyperparameters_freeze_backbone` | 34.44% | 34.50% | 2.7034 | 68 | 51,300 |
| `resnet18_same_hyperparameters_whole_model` | 71.95% | **72.13%** | **1.3300** | 65 | 11,227,812 |
| `resnet18_best` (saved JSON configuration) | 68.02% | 68.86% | 1.5268 | 64 | 11,227,812 |

The strongest recorded ResNet result in the notebook is therefore `resnet18_same_hyperparameters_whole_model`, not the run named `resnet18_best`. It is the only recorded run that exceeds the assignment's approximate 70% target. The `resnet18_best` source/output mismatch is documented in its own explanation and should be resolved by rerunning under a single declared configuration before treating it as a reproducible final experiment.

# Final remarks

The custom residual CNN satisfies the Part 1 target. The preserved final compact `best_model` reaches **56.62% test accuracy** with **2,824,580 parameters**. The earlier large base sweep reaches as high as **59.47%**, but uses **11,227,812 parameters**. The notes explicitly choose the compact `[32, 64, 128, 256]` model as a size/accuracy trade-off: roughly one quarter of the parameters for a few percentage points of accuracy.

The most convincing Part 1 evidence concerns the input pipeline and normalization. In the historical controlled base configuration, removing augmentation reduces test accuracy from **57.97% to 10.71%** and creates a large train/validation gap. Removing batch normalization reduces test accuracy to **48.96%**, and reducing the maximum learning rate to `1e-4` reduces it to **40.53%**. Mixup and label smoothing provide smaller gains: their zero-valued historical runs reach **56.74%** and **56.17%**, respectively. Random erasing is not supported as beneficial by the saved single-run comparison, because removing it gives **58.51%** versus the **57.97%** reference.

For ResNet-18, backbone adaptation is the dominant decision. With the Part 1 hyperparameters, freezing the pretrained backbone reaches **34.50%** test accuracy, whereas training the whole model reaches **72.13%** and lowers test loss from **2.7034 to 1.3300**. This controlled result shows that ImageNet features are useful but must be fine-tuned for the fine-grained differences among aircraft variants. The historical differential-learning-rate sweep supports using a smaller backbone rate than head rate, reaching **61.93%** at `1e-4/1e-3`, but it does not surpass the 72.13% whole-model run.

The final model selected by recorded test accuracy is `resnet18_same_hyperparameters_whole_model` at **72.13%**, which meets the approximate Part 2 target. The run called `resnet18_best` records **68.86%** and is not the numerical best. Its saved JSON and current configuration cell disagree, so its preserved result must be associated with the saved values (`lr=2e-4`, `head_lr=1e-3`, `mixup_alpha=0.9`, `mixup_prob=0.3`) until it is rerun.

Finally, these are single-seed experiments. Several requested Part 2B zero-valued ablations do not have directly matching saved artifacts, so their new run cells are included but no outcomes are fabricated. Repeated seeds and execution of those exact current-model ablations would be needed to distinguish small hyperparameter effects from run-to-run variation.